In [1]:
cd ..

/Users/matthewshen/Desktop/drd


In [2]:

import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
import umap
from drd.drd import DRD  # Your sklearn-compatible autoencoder

ModuleNotFoundError: No module named 'sklearn'

In [ ]:

# Load Dataset (Wine dataset - small but structured)
data = load_wine()
X = data.data
y = data.target

# Standardize Data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Create UMAP Embedding (Teacher)
umap_model = umap.UMAP(n_components=2, random_state=42)
X_umap = umap_model.fit_transform(X_scaled)

# Convert Teacher Embedding to Torch Tensor
X_umap_tensor = torch.tensor(X_umap, dtype=torch.float32)

# Initialize & Train the Student Model (DRD AutoEncoder)
student_model = DRD(input_dim=X_scaled.shape[1], latent_dim=2, lambda_kl = 0.1, epochs=100, batch_size=16)

# Fit the DRD model using UMAP embedding as the teacher
student_model.fit(X_scaled, teacher_model=X_umap_tensor)

# Extract Learned Embeddings
X_student = student_model.transform(X_scaled)

# Plot UMAP vs. DRD Learned Embedding
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# UMAP Visualization
axes[0].scatter(X_umap[:, 0], X_umap[:, 1], c=y, cmap='viridis', alpha=0.7)
axes[0].set_title("UMAP Embedding (Teacher)")

# DRD Student Model Visualization
axes[1].scatter(X_student[:, 0], X_student[:, 1], c=y, cmap='viridis', alpha=0.7)
axes[1].set_title("DRD Learned Embedding (Student)")

plt.show()
